# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [3]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('/Users/gayathrinarayanan/Desktop/week 07 /data/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [4]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [9]:
# Task 1
# YOUR CODE HERE

df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

target_ratings = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']
filtered = df[df['rating'].isin(target_ratings)]

pivot = (
    filtered.groupby(['rating', 'decade'])
    .size()
    .reset_index(name='count')
    .pivot(index='rating', columns='decade', values='count')
    .fillna(0)
    .astype(int)
)
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

fig = px.imshow(
    pivot,
    color_continuous_scale='RdYlGn',
    text_auto=True,
    aspect='auto',
    height=450, width=900,
)

fig.update_traces(
    textfont=dict(size=12, color='black'),
    hovertemplate='<b>%{y}</b> — %{x}<br>Titles: %{z}<extra></extra>',
)

import numpy as np

# Peak cell
flat_peak = pivot.values.argmax()
peak_pos = np.unravel_index(flat_peak, pivot.shape)
peak_val = pivot.values[peak_pos]
peak_rating = pivot.index[peak_pos[0]]
peak_decade = pivot.columns[peak_pos[1]]

# Lowest non-zero cell
nonzero = pivot.replace(0, np.nan)
flat_low = np.nanargmin(nonzero.values)          # flat index into the 2D array
low_pos = np.unravel_index(flat_low, pivot.shape)  # convert to (row, col)
low_val = pivot.values[low_pos]
low_rating = pivot.index[low_pos[0]]
low_decade = pivot.columns[low_pos[1]]

fig.update_layout(
    title=f'TV-MA dominates the 2010s–2020s | Peak: {peak_rating} in {peak_decade} ({peak_val}) | Lowest: {low_rating} in {low_decade} ({low_val})',
    font=dict(family='Arial', size=11),
    coloraxis_colorbar=dict(title='Titles', thickness=15),
    margin=dict(l=80, r=40, t=70, b=60),
)
fig.update_xaxes(title='Release Decade')
fig.update_yaxes(title='')

# Black border on peak cell
fig.add_shape(
    type='rect',
    x0=peak_pos[1] - 0.5, x1=peak_pos[1] + 0.5,
    y0=peak_pos[0] - 0.5, y1=peak_pos[0] + 0.5,
    line=dict(color='black', width=3),
)

# Blue border on lowest cell
fig.add_shape(
    type='rect',
    x0=low_pos[1] - 0.5, x1=low_pos[1] + 0.5,
    y0=low_pos[0] - 0.5, y1=low_pos[0] + 0.5,
    line=dict(color='blue', width=3),
)

fig.show()

## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [6]:
# Task 2
# YOUR CODE HERE

# Task 2
movies = df[df['type'] == 'Movie'].copy()
movies['added_year'] = pd.to_numeric(movies['added_year'], errors='coerce')
movies = movies.dropna(subset=['added_year'])
movies['added_year'] = movies['added_year'].astype(int)

adds = (
    movies[movies['added_year'].between(2015, 2022)]
    .groupby('added_year')
    .size()
    .reset_index(name='new_titles')
)

adds['added_year'] = adds['added_year'].astype(str)

cumulative_total = adds['new_titles'].sum()
peak_idx = adds['new_titles'].idxmax()
peak_year = adds.loc[peak_idx, 'added_year']
peak_count = adds.loc[peak_idx, 'new_titles']

x_vals   = adds['added_year'].tolist() + ['Total 2015–2022']
y_vals   = adds['new_titles'].tolist() + [cumulative_total]
measures = ['relative'] * len(adds) + ['total']
bar_text = [f'{v:,}' for v in adds['new_titles']] + [f'{cumulative_total:,}']

trace = go.Waterfall(
    x=x_vals,
    y=y_vals,
    measure=measures,
    text=bar_text,
    textposition='outside',
    textfont=dict(size=11, color='#333333'),
    connector=dict(line=dict(color='#AAAAAA', dash='dot')),
    increasing=dict(marker_color='#70AD47'),
    totals=dict(marker_color='#2E75B6'),
)

fig = go.Figure(data=[trace])

fig.add_annotation(
    x=peak_year,
    y=peak_count + 30,
    text=f'<b>Peak year</b><br>{peak_year}: {peak_count:,} movies',
    showarrow=True,
    arrowhead=2,
    ax=0, ay=-45,
    font=dict(size=11, color='#333333'),
    bgcolor='#FFFDE7',
    bordercolor='#CCCCCC',
    borderwidth=1,
)

fig.update_layout(
    title='Netflix nearly doubled its Movie catalogue between 2015 and 2019, then plateaued',
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    showlegend=False,
    margin=dict(l=60, r=40, t=60, b=50),
    height=550,
)
fig.update_xaxes(title='Year', showgrid=False)
fig.update_yaxes(title='Movies Added', gridcolor='#EEEEEE')
fig.show()

   